# 07. Model Shootout: Binario vs Multiclase\nNotebook temporal donde comprobamos el techo matemático del dataset y justificamos la migración hacia una estrategia Binaria con un Sistema Experto Híbrido, descartando el Random Forest Multiclase.\n

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, f1_score, precision_recall_curve
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Cargar y Transformar Datos a Binario
print("Cargando datos y transformando a problema binario...")
df = pd.read_csv('../data/03_primary/tabla_maestra.csv')
df['estado_matricula'] = df['estado_matricula'].str.strip().str.upper()

# Agrupamos las categorías
df['estado_matricula'] = df['estado_matricula'].replace(['CONGELADA', 'DESERTOR'], 'BAJA_RETENCION')
df['estado_matricula'] = df['estado_matricula'].replace(['REGULAR', 'EGRESADO'], 'NO_RIESGO')

# Mapeo numérico (1 = Riesgo, 0 = No Riesgo)
df['target_bin'] = df['estado_matricula'].map({'NO_RIESGO': 0, 'BAJA_RETENCION': 1})

num_cols = ["total_ausencias", "promedio_notas", "semestre"]
cat_cols = ["carrera", "sede"]
target_col = "target_bin"

df_ml = df.dropna(subset=num_cols + cat_cols + [target_col]).copy()
X = df_ml[num_cols + cat_cols]
y = df_ml[target_col]

# Partición de datos con Semilla Óptima Encontrada (44)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=44, stratify=y
)

### Comprobación Final del Híbrido\n

In [ ]:
print("Pipeline de Shootout: SMOTEENN + LGBM Binario")
pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smoteenn', SMOTEENN(random_state=42)),
    ('classifier', LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1))
])
pipe.fit(X_train, y_train)

### Resultados\n

In [ ]:
# Buscador Algorítmico del Umbral Óptimo Matemático
y_probs = pipeline.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

f1_scores = [0.0 if (p + r) == 0 else 2 * (p * r) / (p + r) for p, r in zip(precisions[:-1], recalls[:-1])]
best_threshold = thresholds[np.argmax(f1_scores)]

y_pred_base = (y_probs >= best_threshold).astype(int)

# Sistema Experto Híbrido: Regla de Salvación (Override)
y_pred_expert = y_pred_base.copy()

# Regla: Si predice Riesgo (1), pero el promedio es >= 6.0, forzar a No Riesgo (0)
mask_override = (y_pred_expert == 1) & (X_test['promedio_notas'] >= 6.0)
y_pred_expert[mask_override] = 0

print("=== Classification Report Final ===")
print(classification_report(y_test, y_pred_expert, target_names=['NO_RIESGO (0)', 'BAJA_RETENCION (1)']))